In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/small_bench/checkpoints/pre_cell_43.pickle

In [ ]:
%%PandasProfile
### cell 43 ###

def count_then_return_percent_55(df, col):
    # Compute value counts (including NaNs) once and reuse the total count
    total = df[col].count()
    return (df[col]
            .value_counts(dropna=False)
            .mul(100)
            .div(total)
            .round(1)
           )


def combine_subset_of_data_from_multiple_years_55(question, x_axis_title, include_2017=None):
    # Map each year to its response dataframe
    mapping = {
        '2022': responses_df_2022_cell10,
        '2021': responses_df_2021,
        '2020': responses_df_2020,
        '2019': responses_df_2019_cell10,
        '2018': responses_df_2018_cell10,
    }
    if include_2017 is not None:
        mapping['2017'] = responses_df_2017

    # Iterate in sorted year order, compute percentages, and build a list of small DataFrames
    dfs = []
    for year in sorted(mapping.keys()):
        ser = count_then_return_percent_55(mapping[year], question).sort_index()
        df_temp = pd.DataFrame({
            'percentage': ser,
            'year': year,
        })
        # Preserve the category as index and also as a column
        df_temp[x_axis_title] = ser.index
        dfs.append(df_temp)

    # Concatenate all years into one DataFrame
    df_combined = pd.concat(dfs)
    # Ensure column order
    return df_combined[['percentage', 'year', x_axis_title]]

# Usage remains unchanged
question_of_interest_cell55 = (
    'Select the title most similar to your current role (or most recent title if retired): '
    '- Selected Choice'
)
job_df_combined = combine_subset_of_data_from_multiple_years_55(
    question_of_interest_cell55,
    title_for_x_axis_cell43
)
# Replace and filter as before
job_df_combined.replace(
    ['Data Analyst (Business, Marketing, Financial, Quantitative, etc)'],
    'Data Analyst',
    inplace=True
)
job_df_combined_cell55 = job_df_combined[
    job_df_combined.index.isin([
        'Data Analyst', 'Data Engineer', 'Data Scientist',
        'Research Scientist', 'Software Engineer'
    ])
]
job_df_combined_cell55.info()
